# 02 - Sentiment Exploration

Loads the sample news/social dataset, scores it with the configured sentiment model (FinBERT by default), and aggregates it into the configured time window. See `src/sentiment/` for the implementation and the README's "Sentiment Model" section for the FinBERT-vs-generic trade-off discussion.

Note: the first run of this notebook will download the FinBERT weights (~440MB) from Hugging Face if they are not already cached locally.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import SAMPLE_DATA_DIR, SentimentConfig
from src.data.loader import load_market_data_csv, load_news_data_json
from src.sentiment.aggregator import aggregate_sentiment_to_windows
from src.sentiment.preprocessing import preprocess_news_dataframe
from src.sentiment.sentiment_model import SentimentScorer

%matplotlib inline

In [ ]:
news_df = load_news_data_json(SAMPLE_DATA_DIR / "news_data.json")
print(news_df.shape)
news_df.head()

In [ ]:
cleaned_news = preprocess_news_dataframe(news_df)
cleaned_news.head()

## Score with FinBERT (default) and compare with the generic model

In [ ]:
finbert_scorer = SentimentScorer(SentimentConfig(model_family="finbert"))
scored_finbert = finbert_scorer.score_dataframe(cleaned_news)
scored_finbert[['timestamp', 'text', 'sentiment_label', 'sentiment_score']].head(10)

In [ ]:
generic_scorer = SentimentScorer(SentimentConfig(model_family="generic"))
scored_generic = generic_scorer.score_dataframe(cleaned_news)

comparison = pd.DataFrame({
    'text': cleaned_news['text'],
    'finbert_label': scored_finbert['sentiment_label'],
    'finbert_score': scored_finbert['sentiment_score'],
    'generic_label': scored_generic['sentiment_label'],
    'generic_score': scored_generic['sentiment_score'],
})
disagreements = comparison[comparison['finbert_label'] != comparison['generic_label']]
print(f"{len(disagreements)} / {len(comparison)} items where FinBERT and the generic model disagree")
disagreements.head(10)

## Sentiment label distribution

In [ ]:
scored_finbert['sentiment_label'].value_counts().plot(kind='bar', title='FinBERT sentiment label distribution')
plt.show()

## Aggregate into hourly windows aligned with market data

In [ ]:
market_df = load_market_data_csv(SAMPLE_DATA_DIR / "market_data.csv")
aggregated = aggregate_sentiment_to_windows(scored_finbert, market_df['timestamp'], window="1h")
aggregated.describe()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(market_df['timestamp'], aggregated['mean_sentiment'])
plt.title('Mean aggregated sentiment per hourly window')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.show()